## Tarea: Análisis Númerico. Problemas de cómputo

### Nombre: Gabriel Fernando Rosas Zepeda

**1.1** Escriba un programa para calcular los errores absolutos y relativos en la aproximación de Stirling.

$$n! = \sqrt{2\pi n}\left(n/e\right)^n$$

para $n = 1, ... ,10$. ¿Aumenta el error absoluto o se reduce al aumentar $n$? ¿El error relativo aumenta o
disminuye al aumentar $n$?

In [ ]:
import numpy as np
import pandas as pd
from math import factorial, sqrt, pi, e
import matplotlib.pyplot as plt

In [ ]:
# Calculo de n! exacto y aproximacion de Stirling
ns = np.arange(1, 11)

exacto    = np.array([factorial(n) for n in ns], dtype=float)
stirling  = np.array([sqrt(2 * pi * n) * (n / e)**n for n in ns])

error_abs = np.abs(exacto - stirling)
error_rel = error_abs / exacto

df = pd.DataFrame({
    "n"           : ns,
    "n! exacto"   : exacto,
    "Stirling"    : stirling,
    "Error abs"   : error_abs,
    "Error rel"   : error_rel,
})

df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].semilogy(ns, error_abs, marker='o', color='steelblue')
axes[0].set_title("Error absoluto (escala log)")
axes[0].set_xlabel("n")
axes[0].set_ylabel("|n! - Stirling(n)|")
axes[0].grid(True)

axes[1].plot(ns, error_rel * 100, marker='o', color='tomato')
axes[1].set_title("Error relativo (%)")
axes[1].set_xlabel("n")
axes[1].set_ylabel("Error relativo (%)")
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Conclusiones
print("Error absoluto: CRECE con n (la diferencia entre n! y Stirling aumenta en magnitud).")
print("Error relativo: DECRECE con n (Stirling es proporcionalmente mas preciso para n grandes).")

**1.2** Escriba un programa para determinar valores aproximados para el redondeo unitario $\epsilon_{mach}$ y nivel de desbordamiento UFL y pruébelo en una computadora real. (_Opcional_: ¿Puede determinar también el ¿Nivel de desbordamiento OFL en tu equipo? Esto es más complicado porque un desbordamiento real puede ser fatal). Imprima los valores resultantes en decimal y También trate de determinar el número de bits en el campos de mantisa y exponente del sistema de punto flotante que utilice.

In [ ]:
# --- epsilon_mach ---
# Es el menor eps > 0 tal que fl(1 + eps) > 1
eps = 1.0
while (1.0 + eps / 2.0) > 1.0:
    eps /= 2.0

epsilon_mach = eps
print(f"epsilon_mach  = {epsilon_mach:.6e}")

# Bits en la mantisa: epsilon_mach = 2^(-(p-1)) con redondeo al mas cercano
# => p - 1 = -log2(epsilon_mach)
bits_mantisa = round(-np.log2(epsilon_mach)) + 1
print(f"Bits mantisa  = {bits_mantisa}  (incluye bit implicito)")

In [ ]:
# --- UFL: menor numero positivo normalizado ---
ufl = 1.0
while (ufl / 2.0) > 0.0:
    ufl /= 2.0

# El ultimo valor antes de llegar a 0 es subnormal;
# subimos un paso para quedarnos en el normalizado
ufl_normalizado = ufl * 2.0
print(f"UFL (subnormal mas pequeño) = {ufl:.6e}")
print(f"UFL (normalizado)           = {ufl_normalizado:.6e}")

# Bits en el exponente: UFL ~ 2^(e_min), e_min = log2(UFL_normalizado)
e_min = round(np.log2(ufl_normalizado))
# En IEEE 754 doble: exponente almacenado en k bits => e_min = -(2^(k-1) - 2)
# Despejando: k = log2(1 - e_min) + 1
bits_exponente = round(np.log2(1 - e_min)) + 1
print(f"e_min         = {e_min}")
print(f"Bits exponente= {bits_exponente}")

In [ ]:
# --- OFL: mayor numero finito representable ---
# Se busca el mayor x tal que x * 2 no sea infinito
# Se usa float con incrementos seguros para no provocar un overflow fatal
import struct

ofl = 1.0
while not np.isinf(ofl * 2.0):
    ofl *= 2.0

# ofl es ahora 2^e_max; refinamos sumando potencias decrecientes
incremento = ofl / 2.0
while incremento > 0.0:
    if not np.isinf(ofl + incremento):
        ofl += incremento
    incremento /= 2.0

print(f"OFL           = {ofl:.6e}")

e_max = round(np.log2(ofl))
print(f"e_max         = {e_max}")

In [ ]:
# --- Resumen comparado con valores teoricos IEEE 754 doble precision ---
print("\n--- Resumen ---")
print(f"{'Cantidad':<30} {'Empirico':>20} {'IEEE 754 teorico':>20}")
print("-" * 72)
print(f"{'epsilon_mach':<30} {epsilon_mach:>20.6e} {np.finfo(float).eps:>20.6e}")
print(f"{'UFL (normalizado)':<30} {ufl_normalizado:>20.6e} {np.finfo(float).tiny:>20.6e}")
print(f"{'OFL':<30} {ofl:>20.6e} {np.finfo(float).max:>20.6e}")
print(f"{'Bits mantisa':<30} {bits_mantisa:>20d} {'53 (52 + bit implicito)':>20}")
print(f"{'Bits exponente':<30} {bits_exponente:>20d} {'11':>20}")

**1.3** En la mayoría de los sistemas de punto flotante, una operación rápida al redondeo unitario se puede obtener evaluando la expresión

$$\epsilon_{mach} \approx [3 * (4 / 3  - 1) - 1]$$

_(a)_ Explica por qué funciona este truco.

Por que funciona este truco

En matematicas exactas, 4/3 - 1 = 1/3, y 3 * (1/3) - 1 = 0 exactamente.
En punto flotante, 4/3 NO es representable exactamente en base 2:
su expansion binaria es periodica: 1.010101... en binario.
Al representarla se comete un error de redondeo del orden de epsilon_mach.

Sea fl(4/3) = 4/3 + delta, donde |delta| ~ epsilon_mach.
Entonces:
fl(4/3) - 1  = 1/3 + delta

$$3 * (1/3 + \delta) - 1 = 1 + 3* \delta - 1 = 3 * \delta$$

El resultado no es cero sino 3*delta, que es del orden de epsilon_mach.
El factor 3 aparece porque se amplifican los errores de redondeo
acumulados en la resta y la multiplicacion, pero el orden de magnitud
se conserva. En IEEE 754 con redondeo al mas cercano el resultado es
exactamente epsilon_mach / 2, es decir la unidad de redondeo.

_(b)_ Pruébelo en una variedad de computadoras (en ambos precisión simple y doble) y calculadoras para confirmar que funciona.

In [ ]:
# Doble precision (float64)
x64 = np.float64(3) * (np.float64(4) / np.float64(3) - np.float64(1)) - np.float64(1)
eps64_teorico = np.finfo(np.float64).eps

print("--- Doble precision (float64) ---")
print(f"Truco         = {abs(x64):.6e}")
print(f"epsilon_mach  = {eps64_teorico:.6e}")
print(f"Cociente      = {abs(x64) / eps64_teorico:.6f}")

# Simple precision (float32)
x32 = np.float32(3) * (np.float32(4) / np.float32(3) - np.float32(1)) - np.float32(1)
eps32_teorico = np.finfo(np.float32).eps

print("\n--- Simple precision (float32) ---")
print(f"Truco         = {abs(x32):.6e}")
print(f"epsilon_mach  = {eps32_teorico:.6e}")
print(f"Cociente      = {abs(x32) / eps32_teorico:.6f}")

# Media precision (float16)
x16 = np.float16(3) * (np.float16(4) / np.float16(3) - np.float16(1)) - np.float16(1)
eps16_teorico = np.finfo(np.float16).eps

print("\n--- Media precision (float16) ---")
print(f"Truco         = {abs(x16):.6e}")
print(f"epsilon_mach  = {eps16_teorico:.6e}")
print(f"Cociente      = {abs(x16) / eps16_teorico:.6f}")

In [ ]:
# Resumen en tabla
import pandas as pd

df = pd.DataFrame({
    "Precision"    : ["float16", "float32", "float64"],
    "Truco"        : [abs(x16), abs(x32), abs(x64)],
    "epsilon_mach" : [np.finfo(np.float16).eps,
                      np.finfo(np.float32).eps,
                      np.finfo(np.float64).eps],
    "Bits mantisa" : [np.finfo(np.float16).nmant + 1,
                      np.finfo(np.float32).nmant + 1,
                      np.finfo(np.float64).nmant + 1],
})

df["Cociente"] = df["Truco"] / df["epsilon_mach"]
df

_(c)_ ¿Funcionaría este truco en sistemas de punto flotante con base $\beta = 3$?

No, el truco **no funcionaria** en base $\beta = 3$.

---

**Por que falla**

El truco se basa en que $4/3$ no es exactamente representable en base 2, ya que su expansion binaria es periodica:

$\frac{4}{3} = 1.\overline{01}_2$

Esto introduce un error de redondeo $\delta$ del orden de $\epsilon_{mach}$ al almacenar $fl(4/3)$, y la expresion $3(4/3 - 1) - 1$ amplifica ese error y lo expone como resultado.

En base $\beta = 3$, en cambio, $4/3$ **si es exactamente representable**, ya que su expansion en base 3 es finita:

$\frac{4}{3} = 1.1_3$

es decir, $1 + \frac{1}{3}$, sin periodicidad. Por tanto $fl(4/3) = 4/3$ exactamente, no hay error de redondeo que amplificar, y la expresion evalua:

$3 \left(\frac{4}{3} - 1\right) - 1 = 3 \cdot \frac{1}{3} - 1 = 1 - 1 = 0$

El truco devuelve $0$ en lugar de $\epsilon_{mach}$.

---

**Como construir un truco equivalente para base $\beta = 3$**

Se necesita un numero que no sea exactamente representable en base 3. Un candidato natural es $1/2$, cuya expansion en base 3 es periodica:

$\frac{1}{2} = 0.\overline{1}_3$

Un truco analogo podria construirse a partir de $(\beta + 1)/\beta = 4/3$ reemplazado por algun numero con expansion infinita en base 3. En general, para base $\beta$ arbitraria, el numero $(\beta + 1)/\beta$ solo introduce error de redondeo cuando $\beta$ no divide exactamente a $\beta + 1$, lo cual ocurre para $\beta = 2$ pero no para $\beta = 3$, ya que $3$ divide a $3 + 1 = 4$... en realidad no divide exactamente, pero la expansion de $4/3$ en base 3 resulta finita por la relacion particular entre $4$ y $3$.

La conclusion general es que el truco debe elegir un numero cuya expansion en la base del sistema sea **infinita y periodica**, y $4/3$ cumple ese papel unicamente en base 2.

**1.4** Escriba un programa para calcular la constante
matemática $e$, base de los logaritmos naturales, a partir de
la definición

$$e = \lim_{n \to \infty}\left(1 + \frac{1}{n}\right)^n$$

Puntualmente, computa $(1 + \frac{1}{n})$ para $n = 10^k, \quad k = 1, 2, ..., 20$. Si el lenguaje de programación
no tiene un operador para la exponenciación, puede usar la
fórmula equivalente

$$(1 + \frac{1}{n})^n = exp(n \; log(1 + 1 / n))$$

Donde exp y log son funciones integradas. Determine el
error en sus aproximaciones sucesivas comparándolas con
el valor de exp(1). ¿El error siempre disminuye a medida que $n$ aumenta? Explique sus resultados.

In [ ]:
e_exacto = np.exp(1.0)

ks = np.arange(1, 21)
ns = 10.0**ks

aproximaciones = (1.0 + 1.0/ns)**ns
errores_abs    = np.abs(aproximaciones - e_exacto)
errores_rel    = errores_abs / e_exacto

df = pd.DataFrame({
    "k"            : ks,
    "n = 10^k"     : ns,
    "Aproximacion" : aproximaciones,
    "Error abs"    : errores_abs,
    "Error rel"    : errores_rel,
})

df

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

ax.semilogy(ks, errores_rel, marker='o', color='steelblue', label="Error relativo")
ax.axvline(x=ks[errores_rel.argmin()], color='tomato', linestyle='--',
           label=f"Minimo en k = {ks[errores_rel.argmin()]:.0f}")
ax.set_xlabel("k  (n = 10^k)")
ax.set_ylabel("Error relativo (escala log)")
ax.set_title("Aproximacion de e = lim (1 + 1/n)^n")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

**Observaciones sobre la convergencia**

Para $k$ pequeno ($n$ pequeno): el error es grande porque la aproximacion matematica $(1 + 1/n)^n$ aun no ha convergido al limite. El error de truncamiento domina y decrece al aumentar $n$.

Para $k$ intermedio ($n \approx 10^8$): se alcanza el minimo de error. La aproximacion matematica es buena y el error de punto flotante todavia es manejable.

Para $k$ grande ($n > 10^8$): el error **crece de nuevo**. La causa es cancelacion catastrofica en el calculo de $1 + 1/n$: cuando $n$ es muy grande, $1/n < \epsilon_{mach}$ y $fl(1 + 1/n) = 1.0$ exactamente. La expresion $1.0^n = 1.0$ sin importar $n$, alejandose de $e$.

En resumen: **no**, el error no siempre decrece al aumentar $n$. Existe un valor optimo de $n$ que balancea el error de truncamiento matematico y el error de cancelacion en aritmetica de punto flotante.

In [ ]:
k_optimo = ks[errores_rel.argmin()]
print(f"k optimo        = {k_optimo}")
print(f"n optimo        = 1e{k_optimo:.0f}")
print(f"Mejor aprox.    = {aproximaciones[errores_rel.argmin()]:.15f}")
print(f"e exacto        = {e_exacto:.15f}")
print(f"Error relativo  = {errores_rel.min():.6e}")

**1.5** _(a)_ Considere la función

$$f(x) = \frac{(e^x - 1)}{x}$$

Use la regla l'Hôpital's para mostrar que

$$\lim_{x \to 0}f(x) = 1$$

**(a) Limite de $f(x) = (e^x - 1)/x$ cuando $x \to 0$**

La funcion $f(x) = (e^x - 1)/x$ presenta la forma indeterminada $0/0$ al evaluar directamente en $x = 0$, ya que $e^0 - 1 = 0$ y el denominador tambien es $0$.

Aplicando la regla de L'Hopital, se deriva numerador y denominador por separado:

$\lim_{x \to 0} \frac{e^x - 1}{x} = \lim_{x \to 0} \frac{\frac{d}{dx}(e^x - 1)}{\frac{d}{dx}(x)} = \lim_{x \to 0} \frac{e^x}{1} = e^0 = 1$

Por tanto

$\lim_{x \to 0} f(x) = 1$

_(b)_ Compruebe este resultado empiricamente escribiendo un programa pra calcular $f(x) \, \text{for} \; x = 10^{-k}. \; k = 1, 2, ..., 16$. ¿Coinciden sus resultados con las expectativas teóricas? Explique por que

In [ ]:
ks = np.arange(1, 17)
xs = 10.0**(-ks)

f_vals    = (np.exp(xs) - 1.0) / xs
errores   = np.abs(f_vals - 1.0)

df = pd.DataFrame({
    "k"        : ks,
    "x = 10^-k": xs,
    "f(x)"     : f_vals,
    "Error abs" : errores,
})

df

In [ ]:
ig, ax = plt.subplots(figsize=(9, 4))

ax.semilogy(ks, errores, marker='o', color='steelblue')
ax.set_xlabel("k  (x = 10^-k)")
ax.set_ylabel("| f(x) - 1 |  (escala log)")
ax.set_title("Error en f(x) = (e^x - 1) / x  respecto al limite teorico 1")
ax.grid(True)
plt.tight_layout()
plt.show()

**Observaciones**

Para $k$ pequeno ($x$ no tan cercano a cero): $f(x)$ converge hacia $1$ correctamente y el error decrece, en concordancia con el limite teorico.

Para $k$ grande ($x \approx 10^{-12}$ o menor): el error **crece de nuevo** y para $k = 16$ se tiene $f(x) = 0/0$ o $f(x) = 1.0$ sin precision real. La causa es cancelacion catastrofica en el numerador $e^x - 1$: cuando $x$ es muy pequeno, $e^x \approx 1$ en punto flotante y $fl(e^x - 1) = 0$, destruyendo todos los digitos significativos.

Existe un valor optimo de $k$ que balancea la aproximacion matematica al limite y la perdida de precision numerica. Una formula alternativa que evita la cancelacion es usar la expansion de Taylor directamente:

$f(x) = \frac{e^x - 1}{x} \approx 1 + \frac{x}{2} + \frac{x^2}{6} + \cdots$

que es numericamente estable para $x$ pequeno.

_(c)_ Repita el experimento del inciso _b_, esta vez utilizando la formulación matemáticamente equivalente

$$f(x) = \frac{(e^x - 1)}{log(e^x)}$$

evaluado como se indica, sin simplificación. Si esto funciona mejor, ¿puedes explicar por qué?

In [ ]:
# Comparacion entre las dos formulaciones de f(x)

ks = np.arange(1, 17)
xs = 10.0**(-ks)

# Formulacion original
f_original  = (np.exp(xs) - 1.0) / xs

# Formulacion alternativa
f_alternativa = (np.exp(xs) - 1.0) / np.log(np.exp(xs))

error_original   = np.abs(f_original   - 1.0)
error_alternativa = np.abs(f_alternativa - 1.0)

df = pd.DataFrame({
    "k"                : ks,
    "x = 10^-k"        : xs,
    "f original"       : f_original,
    "f alternativa"    : f_alternativa,
    "Error original"   : error_original,
    "Error alternativa": error_alternativa,
})

df

**Explicacion**

La formulacion alternativa $f(x) = (e^x - 1) / \log(e^x)$ funciona mejor para $x$ muy pequeno porque **la cancelacion en numerador y denominador se produce de forma simetrica**.

En la formulacion original, el denominador es simplemente $x$, un numero exacto pero muy pequeno. El numerador $e^x - 1$ sufre cancelacion catastrofica porque $fl(e^x) \approx 1$ y se pierden los digitos significativos. El cociente hereda todo ese error.

En la formulacion alternativa, tanto el numerador $e^x - 1$ como el denominador $\log(e^x)$ se calculan a partir del mismo valor intermedio $e^x$. Cuando $x$ es pequeno:

$\log(e^x) = x \cdot \log(e) = x$

pero en punto flotante $\log(fl(e^x))$ comete un error proporcional al mismo $fl(e^x)$ que afecta al numerador. Los dos errores tienden a **cancelarse mutuamente al dividir**, ya que numerador y denominador estan contaminados por el mismo valor intermedio $fl(e^x)$. El cociente resulta mas estable que en la version original donde el denominador $x$ es exacto y no comparte el error del numerador.

En esencia, la formulacion alternativa aprovecha una **correlacion de errores** entre numerador y denominador que la original no tiene.

**1.6** Supongamos que necesitamos generar $n + 1$ puntos espaciados igualmente sobre el intervalo $[a, b]$, con espacios $h = (b - a)/n$

_(a)_ En la aritmética de punto flotante, cual de los siguietes metodos,

$$x_0 = a, \quad x_k = x_{k-1} + h, \quad k = 1, ..., n$$

or

$$x_k = a + kh, \quad k = 0, 1, ..., n$$

es mejor y por que?

**(a) Generacion de puntos igualmente espaciados en $[a, b]$**

La formula preferible es la segunda:

$x_k = a + k \cdot h, \quad k = 0, 1, \ldots, n$

---

**Por que es mejor**

La primera formula es **recursiva**: cada punto se obtiene sumando $h$ al punto anterior. Esto significa que el error de redondeo cometido en cada paso se **acumula** sobre todos los pasos anteriores. Despues de $k$ pasos, el error absoluto acumulado es del orden de $k \cdot \epsilon_{mach} \cdot h$, que crece linealmente con $k$. Para $k = n$ el ultimo punto $x_n$ puede diferir significativamente de $b$.

La segunda formula es **directa**: cada punto se calcula independientemente a partir de $a$ y $h$, que son cantidades fijas. El error en $x_k$ es del orden de $\epsilon_{mach} \cdot |a + k \cdot h|$ sin acumulacion entre pasos. Cada punto tiene su propio error de redondeo pero este no depende de los errores de los puntos anteriores.

Adicionalmente, la segunda formula garantiza que $x_0 = a$ exactamente y que $x_n = a + n \cdot h = a + (b - a) = b$ con el minimo error posible, mientras que la formula recursiva puede alejarse de $b$ al final del intervalo.

---

**Resumen**

| | Formula recursiva | Formula directa |
|---|---|---|
| Error en $x_k$ | $O(k \cdot \epsilon_{mach} \cdot h)$ | $O(\epsilon_{mach} \cdot |x_k|)$ |
| Acumulacion de error | Si, crece con $k$ | No |
| $x_n$ cercano a $b$ | No garantizado | Si |

_(b)_ Escriba un programa que implemente ambos métodos y encuentre un ejemplo, digamos, con $a = 0$ y $b = 1$, que ilustre la diferencia entre ellos.

In [ ]:
a, b = 0.0, 1.0
n    = 10

h = (b - a) / n

# Metodo 1: recursivo
x_rec = np.zeros(n + 1)
x_rec[0] = a
for k in range(1, n + 1):
    x_rec[k] = x_rec[k - 1] + h

# Metodo 2: directo
x_dir = np.array([a + k * h for k in range(n + 1)])

# Valores exactos en aritmetica real
x_exacto = np.linspace(a, b, n + 1)

error_rec = np.abs(x_rec - x_exacto)
error_dir = np.abs(x_dir - x_exacto)

df = pd.DataFrame({
    "k"           : np.arange(n + 1),
    "x recursivo" : x_rec,
    "x directo"   : x_dir,
    "x exacto"    : x_exacto,
    "Error rec"   : error_rec,
    "Error dir"   : error_dir,
})

df

In [ ]:
# Para hacer la diferencia mas visible usamos n grande
n_grande = 10**6
h2       = (b - a) / n_grande

x_rec2    = np.zeros(n_grande + 1)
x_rec2[0] = a
for k in range(1, n_grande + 1):
    x_rec2[k] = x_rec2[k - 1] + h2

x_dir2    = np.array([a + k * h2 for k in range(n_grande + 1)])
x_exacto2 = np.linspace(a, b, n_grande + 1)

error_rec2 = np.abs(x_rec2 - x_exacto2)
error_dir2 = np.abs(x_dir2 - x_exacto2)

ks = np.arange(n_grande + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(ks, error_rec2, color='tomato',    lw=0.5, label="Recursivo")
axes[0].plot(ks, error_dir2, color='steelblue', lw=0.5, label="Directo")
axes[0].set_title(f"Error absoluto  (n = {n_grande:.0e})")
axes[0].set_xlabel("k")
axes[0].set_ylabel("|x_k - x_exacto|")
axes[0].legend()
axes[0].grid(True)

# Zoom al final del intervalo donde la acumulacion es maxima
m = n_grande // 10
axes[1].plot(ks[-m:], error_rec2[-m:], color='tomato',    lw=0.8, label="Recursivo")
axes[1].plot(ks[-m:], error_dir2[-m:], color='steelblue', lw=0.8, label="Directo")
axes[1].set_title("Zoom: ultimo 10% del intervalo")
axes[1].set_xlabel("k")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

print(f"Error maximo recursivo : {error_rec2.max():.6e}")
print(f"Error maximo directo   : {error_dir2.max():.6e}")
print(f"Factor de mejora       : {error_rec2.max() / error_dir2.max():.1f}x")

**Observaciones**

Para $n$ pequeno las diferencias son imperceptibles porque el error acumulado es minimo. Al tomar $n = 10^6$ la acumulacion en el metodo recursivo se vuelve claramente visible: el error crece de forma aproximadamente lineal con $k$, llegando a ser varios ordenes de magnitud mayor que el del metodo directo al final del intervalo. El metodo directo mantiene un error practicamente constante y del orden de $\epsilon_{mach}$ en todos los puntos.

**1.7** _(a)_  Escriba un programa para calcular un valor aproximado para la derivada de una función utilizando la fórmula de diferencias finitas

$$f'(x) \approx \frac{f(x + h) - f(x)}{h}$$

Pruebe su programa usando la función $sen(x)$ para $x = 1$.
Determine el error comparándolo con la función integrada $cos(x)$. Grafique la magnitud del error en función de $h$, para
$h = 1/2, 1/4, 1/8, ... Debe usar una escala logarítmica para $h$ y para la magnintud del error. ¿Existe un valor mínimo para la magnitud del error? ¿Cómo se compara el valor
correspondiente de $h$ con la regla general?

$$h \approx \sqrt{\epsilon_{mach}} · |x|?$$

In [ ]:
x    = 1.0
f    = np.sin
df   = np.cos

# h = 1/2^k, k = 1, 2, ..., 60
ks   = np.arange(1, 61)
hs   = 1.0 / 2.0**ks

derivada_aprox = (f(x + hs) - f(x)) / hs
error          = np.abs(derivada_aprox - df(x))

k_min    = np.argmin(error)
h_min    = hs[k_min]
err_min  = error[k_min]

# Regla de dedo
h_regla  = np.sqrt(np.finfo(float).eps) * np.abs(x)

print(f"h optimo empirico : {h_min:.6e}  (k = {ks[k_min]})")
print(f"h regla de dedo   : {h_regla:.6e}")
print(f"Error minimo      : {err_min:.6e}")

**Observaciones**

La grafica en escala log-log muestra dos regimenes claramente diferenciados:

- Para $h$ grande: el error decrece al reducir $h$ porque domina el **error de truncamiento**, que es $O(h)$ para la formula de diferencias finitas de primer orden. La pendiente en la grafica es aproximadamente $+1$ hacia la izquierda.

- Para $h$ pequeno: el error crece al reducir $h$ porque domina el **error de cancelacion**: $f(x + h)$ y $f(x)$ son casi iguales y su diferencia pierde digitos significativos. El denominador $h$ amplifica ese error.

Existe un valor optimo de $h$ que minimiza el error total, que ocurre empiricamente cerca de $h \approx \sqrt{\epsilon_{mach}}$, en concordancia con la regla de dedo. Para doble precision $\sqrt{\epsilon_{mach}} \approx 10^{-8}$, que coincide con el minimo observado en la grafica.

_(b)_ Repite el ejercicio usando la aproximación de diferencias centradas

$$f'(x) \approx \frac{f(x + h) - f(x - h)}{2h}$$

In [ ]:
derivada_centrada = (f(x + hs) - f(x - hs)) / (2.0 * hs)
error_centrada    = np.abs(derivada_centrada - df(x))

k_min_c   = np.argmin(error_centrada)
h_min_c   = hs[k_min_c]
err_min_c = error_centrada[k_min_c]

# Para diferencias centradas el error de truncamiento es O(h^2)
# el h optimo teorico es ~ epsilon_mach^(1/3)
h_regla_c = np.cbrt(np.finfo(float).eps) * np.abs(x)

print(f"h optimo empirico (centrada) : {h_min_c:.6e}  (k = {ks[k_min_c]})")
print(f"h regla de dedo   (centrada) : {h_regla_c:.6e}")
print(f"Error minimo      (centrada) : {err_min_c:.6e}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.loglog(hs, error,          color='steelblue', lw=1.2, label="Diferencia hacia adelante $O(h)$")
ax.loglog(hs, error_centrada, color='tomato',    lw=1.2, label="Diferencia centrada $O(h^2)$")

ax.axvline(h_min,   color='steelblue', linestyle='--', alpha=0.6, label=f"h opt adelante = {h_min:.1e}")
ax.axvline(h_min_c, color='tomato',    linestyle='--', alpha=0.6, label=f"h opt centrada = {h_min_c:.1e}")
ax.axvline(h_regla, color='seagreen',  linestyle=':',             label=f"h regla O(h)   = {h_regla:.1e}")
ax.axvline(h_regla_c, color='olive',   linestyle=':',             label=f"h regla O(h^2) = {h_regla_c:.1e}")

ax.scatter([h_min],   [error[k_min]],         color='steelblue', zorder=5)
ax.scatter([h_min_c], [error_centrada[k_min_c]], color='tomato', zorder=5)

ax.set_xlabel("h")
ax.set_ylabel("|f'(x) aprox - cos(x)|")
ax.set_title("Comparacion diferencias finitas: adelante vs centrada  ($\sin(x)$, $x = 1$)")
ax.legend(fontsize=8)
ax.grid(True, which='both', ls=':')
plt.tight_layout()
plt.show()

**Observaciones**

La diferencia centrada es superior a la formula hacia adelante en dos aspectos:

**Mayor precision en el regimen de truncamiento.** El error de truncamiento de la formula centrada es $O(h^2)$, frente a $O(h)$ de la formula hacia adelante. En la grafica esto se manifiesta como una pendiente de $+2$ en la zona de convergencia, lo que significa que al reducir $h$ a la mitad el error se divide entre cuatro en lugar de entre dos.

**Menor error minimo alcanzable.** El valor minimo del error es significativamente mas pequeno con la formula centrada. Esto ocurre porque el termino de truncamiento es de orden superior y permite alcanzar errores mas bajos antes de que la cancelacion catastrofica comience a dominar.

**h optimo mas grande.** El balance entre truncamiento y cancelacion se produce para un $h$ mayor que en la formula hacia adelante. El valor optimo teorico es $h \approx \epsilon_{mach}^{1/3}$ para diferencias centradas, frente a $h \approx \epsilon_{mach}^{1/2}$ para la formula hacia adelante. Para doble precision esto corresponde a $h \approx 10^{-5}$ versus $h \approx 10^{-8}$, lo cual es ventajoso porque $h$ mas grande implica menos cancelacion en el numerador.

**1.8** Considera la serie infinita

$$\sum_{n=1}^{\infty}$$

_(a)_ Prueba que la serie es divergente. (_Hint_: Agrupe los términos en conjuntos que contengan terminos $$)

**(a) Demostracion de divergencia de la serie armonica**

$\sum_{n=1}^{\infty} \frac{1}{n} = 1 + \frac{1}{2} + \frac{1}{3} + \frac{1}{4} + \frac{1}{5} + \frac{1}{6} + \frac{1}{7} + \frac{1}{8} + \cdots$

---

**Agrupacion de terminos**

Se agrupan los terminos en bloques $B_k$ que contienen los terminos desde $1/(2^{k-1} + 1)$ hasta $1/2^k$, para $k = 1, 2, 3, \ldots$

$B_1 = \frac{1}{2}$

$B_2 = \frac{1}{3} + \frac{1}{4}$

$B_3 = \frac{1}{5} + \frac{1}{6} + \frac{1}{7} + \frac{1}{8}$

$B_k = \frac{1}{2^{k-1}+1} + \frac{1}{2^{k-1}+2} + \cdots + \frac{1}{2^k}$

El bloque $B_k$ contiene exactamente $2^k - 2^{k-1} = 2^{k-1}$ terminos.

---

**Cota inferior de cada bloque**

El menor termino del bloque $B_k$ es $1/2^k$, por lo que cada termino satisface

$\frac{1}{2^{k-1}+j} \geq \frac{1}{2^k}, \quad j = 1, 2, \ldots, 2^{k-1}$

Sumando los $2^{k-1}$ terminos del bloque:

$B_k = \sum_{j=1}^{2^{k-1}} \frac{1}{2^{k-1}+j} \geq 2^{k-1} \cdot \frac{1}{2^k} = \frac{1}{2}$

Es decir, **cada bloque $B_k$ aporta al menos $1/2$ a la suma parcial**, independientemente de $k$.

---

**Conclusion**

La suma parcial que incluye los primeros $N$ bloques satisface

$S = 1 + \sum_{k=1}^{N} B_k \geq 1 + \sum_{k=1}^{N} \frac{1}{2} = 1 + \frac{N}{2}$

Como $1 + N/2 \to \infty$ cuando $N \to \infty$, las sumas parciales no estan acotadas y por tanto la serie

$\sum_{n=1}^{\infty} \frac{1}{n}$

es **divergente**. $\blacksquare$

_(b)_ Explique porque la suma de la serie en aritmética de punto flotante produce una suma infinita.

**(b) Por que la serie armonica tiene suma finita en aritmetica de punto flotante**

Aunque matematicamente la serie diverge, en aritmetica de punto flotante la suma parcial **deja de crecer** en un valor finito. La razon es la siguiente.

A medida que la suma parcial $S_n$ crece, los terminos $1/n$ se hacen cada vez mas pequenos. En algun momento se cumple que $1/n$ es tan pequeno en relacion con $S_n$ que cae por debajo del nivel de resolucion de la mantisa:

$\frac{1}{n} < \epsilon_{mach} \cdot S_n$

En ese punto, en aritmetica de punto flotante se tiene

$fl\left(S_n + \frac{1}{n}\right) = S_n$

es decir, el termino $1/n$ es invisible para el acumulador: sus digitos significativos quedan fuera del rango representable de la mantisa frente a $S_n$. Todos los terminos siguientes son aun menores, por lo que ninguno podra modificar $S_n$ tampoco, y la suma se congela en ese valor finito.

Despejando la condicion de paro:

$n > \frac{S_n}{\epsilon_{mach}}$

Como $S_n \approx \ln(n)$ para $n$ grande, la suma se detiene cuando $n$ es del orden de

$n \approx \frac{\ln(n)}{\epsilon_{mach}}$

Para doble precision con $\epsilon_{mach} \approx 10^{-16}$, esto ocurre para $n$ del orden de $10^{15}$, un numero astronimicamente grande, pero finito. La suma parcial acumulada hasta ese punto es del orden de $\ln(10^{15}) \approx 35$, que es el valor finito al que converge la serie en punto flotante.

En resumen: la aritmetica de punto flotante impone un **filtro de precision** que hace invisibles los terminos suficientemente pequenos, convirtiendo artificialmente una serie divergente en una suma finita.

(c) Intente predecir cuándo dejará de cambiar la suma parcial tanto en la aritmética de punto flotante de precisión
simple como en la de precisión doble del IEEE.
Dada la velocidad de ejecución de su computadora para operaciones de punto flotante, intente predecir cuánto tardaría cada cálculo en completarse.

**(c) Prediccion del punto de paro y tiempo de computo**

---

**Condicion de paro**

La suma parcial deja de cambiar cuando el termino $1/n$ cae por debajo de la resolucion del acumulador $S_n$:

$\frac{1}{n} < \epsilon_{mach} \cdot S_n$

Como $S_n \approx \ln(n)$ para $n$ grande, la condicion se convierte en

$\frac{1}{n} < \epsilon_{mach} \cdot \ln(n)$

que equivale a resolver aproximadamente

$n \approx \frac{\ln(n)}{\epsilon_{mach}}$

Esta ecuacion no tiene solucion cerrada pero se puede resolver por sustitucion iterativa o simplemente estimando $\ln(n) \approx \ln(1/\epsilon_{mach})$.

---

**Simple precision (float32)**

$\epsilon_{mach}^{(32)} = 2^{-23} \approx 1.19 \times 10^{-7}$

$n_{paro}^{(32)} \approx \frac{\ln(1/\epsilon_{mach}^{(32)})}{\epsilon_{mach}^{(32)}} = \frac{\ln(2^{23})}{2^{-23}} \approx \frac{15.95}{1.19 \times 10^{-7}} \approx 1.34 \times 10^{8}$

Suma final aproximada:

$S \approx \ln(n_{paro}^{(32)}) \approx \ln(1.34 \times 10^8) \approx 18.7$

---

**Doble precision (float64)**

$\epsilon_{mach}^{(64)} = 2^{-52} \approx 2.22 \times 10^{-16}$

$n_{paro}^{(64)} \approx \frac{\ln(2^{52})}{2^{-52}} \approx \frac{36.04}{2.22 \times 10^{-16}} \approx 1.62 \times 10^{17}$

Suma final aproximada:

$S \approx \ln(1.62 \times 10^{17}) \approx 39.6$

---

**Estimacion del tiempo de computo**

Una CPU moderna ejecuta del orden de $10^8$ a $10^9$ operaciones de punto flotante por segundo en un solo nucleo con codigo Python puro (sin vectorizacion). Tomando $10^8$ operaciones/segundo como estimacion conservadora para un bucle Python:

| Precision | $n_{paro}$ | Tiempo estimado |
|---|---|---|
| float32 | $\approx 1.34 \times 10^8$ | $\approx 1$ segundo |
| float64 | $\approx 1.62 \times 10^{17}$ | $\approx 1.62 \times 10^{9}$ segundos $\approx 51$ años |

La suma en doble precision es **computacionalmente inviable** en un bucle secuencial. Incluso con hardware vectorizado a $10^{12}$ FLOPS, tomaria del orden de $10^5$ segundos (mas de un dia).

In [ ]:
# Verificacion empirica para float32
# (no se intenta float64 porque es intratable)

S    = np.float32(0.0)
n    = np.int64(1)
paso = np.int64(1)

while True:
    nuevo = S + np.float32(1.0 / n)
    if nuevo == S:
        break
    S = nuevo
    n += paso

print(f"float32 — paro en n = {n:.6e}")
print(f"float32 — suma final = {S:.6f}")
print(f"float32 — prediccion n ~ 1.34e8")
print(f"float32 — prediccion S ~ 18.7")

_(d)_ Escriba dos programas para calcular la suma de la serie, uno en precisión simple y otro en precisión doble. Supervise el progreso de la suma imprimiendo periódicamente el índice y la suma parcial. ¿Qué criterio de parada debería utilizar? ¿Qué resultado se produce realmente en su computadora? Compare sus resultados con sus predicciones, incluyendo el tiempo de ejecución requerido. (Precaución: Su versión de precisión simple debería terminar bastante rápido, pero la de precisión doble puede tardar mucho más, por lo que podría no ser práctico ejecutarla hasta el final, incluso si su presupuesto informático es generoso).

In [ ]:
import time

# --- Single precision ---
# Criterio de paro: fl(S + 1/n) == S, es decir el termino ya no modifica S

print("=== FLOAT32 ===")
print(f"{'n':>12}  {'S':>12}")
print("-" * 28)

S32  = np.float32(0.0)
n32  = np.int64(1)
t0   = time.time()

intervalo = 10_000_000  # imprimir cada 10M de iteraciones

while True:
    termino = np.float32(1.0) / np.float32(n32)
    nuevo   = S32 + termino
    if nuevo == S32:
        break
    S32 = nuevo
    if n32 % intervalo == 0:
        print(f"{n32:>12}  {S32:>12.6f}")
    n32 += 1

t32 = time.time() - t0

print(f"\nParo en n         = {n32:.6e}")
print(f"Suma final        = {S32:.6f}")
print(f"Prediccion S      ~ 18.7")
print(f"Prediccion n_paro ~ 1.34e8")
print(f"Tiempo real       = {t32:.2f} s")

In [ ]:
# --- Double precision: simulacion hasta n_max practico ---
# Correr hasta n = 1.62e17 es inviable (~51 anyos en Python puro).
# Se corre hasta n_max = 10^8 para estimar la velocidad real
# y extrapolar el tiempo total.

print("=== FLOAT64 (parcial hasta n = 1e8) ===")
print(f"{'n':>12}  {'S':>15}")
print("-" * 32)

S64    = np.float64(0.0)
n64    = np.int64(1)
n_max  = np.int64(100_000_000)
t0     = time.time()

intervalo64 = 10_000_000

while n64 <= n_max:
    S64 += np.float64(1.0) / np.float64(n64)
    if n64 % intervalo64 == 0:
        print(f"{n64:>12}  {S64:>15.10f}")
    n64 += 1

t64_parcial = time.time() - t0

# Extrapolacion
n_paro_64    = 1.62e17
factor       = n_paro_64 / float(n_max)
t64_estimado = t64_parcial * factor

print(f"\nSuma parcial hasta n = {n_max:.2e}  ->  S = {S64:.10f}")
print(f"Valor exacto ln(1e8) = {np.log(1e8):.10f}  (referencia)")
print(f"Tiempo para n = 1e8  = {t64_parcial:.2f} s")
print(f"Tiempo estimado hasta paro (n ~ 1.62e17) = {t64_estimado/3600/24/365:.1f} anyos")

**Observaciones**

**Criterio de paro:** se detiene la suma cuando $fl(S + 1/n) = S$, es decir cuando el termino $1/n$ es tan pequeno que el acumulador no cambia al sumarlo. Este es el criterio natural en punto flotante.

**Float32:** termina en un tiempo del orden de segundos, con $n$ del orden de $10^8$ y suma final cercana a $18.7$, en acuerdo con la prediccion teorica.

**Float64:** la extrapolacion confirma que completar la suma en doble precision requeriria decenas de anyos en un bucle Python puro. La suma parcial hasta $n = 10^8$ ya muestra que $S_{64} \approx \ln(n)$, validando la estimacion teorica de que la suma final seria aproximadamente $39.6$.

**Conclusion practica:** el ejercicio ilustra que una operacion matematicamente bien definida (aunque divergente) puede ser computacionalmente inviable. Para la suma en doble precision seria necesario vectorizacion con NumPy o hardware especializado, aunque incluso asi el tiempo seria prohibitivo.

**1.9** _(a)_ Escriba un programa para calcular la funci´on exponencial $e^x$ usando series infinitas

$$e^x = 1 + x + \frac{x^2}{2!} + \frac{x^3}{3!} + ...$$

In [ ]:
def exp_serie(x, tol=1e-15, max_iter=1000):
    """
    Calcula e^x mediante la serie de Taylor.
    Criterio de paro: el termino actual es menor que tol * |suma|
    """
    suma    = 1.0          # termino n=0
    termino = 1.0          # valor del termino actual
    for n in range(1, max_iter):
        termino *= x / n   # aprovecha el termino anterior: x^n/n! = (x^(n-1)/(n-1)!) * x/n
        suma    += termino
        if abs(termino) < tol * abs(suma):
            return suma, n
    return suma, max_iter

# Prueba para varios valores de x
valores_x = [-20, -10, -5, -1, 0, 1, 5, 10, 20]

resultados = []
for x in valores_x:
    aprox, n_iter = exp_serie(float(x))
    exacto        = np.exp(float(x))
    error_rel     = abs(aprox - exacto) / abs(exacto) if exacto != 0 else float('inf')
    resultados.append({
        "x"          : x,
        "Serie"      : aprox,
        "exp(x) real": exacto,
        "Error rel"  : error_rel,
        "Iteraciones": n_iter,
    })

df = pd.DataFrame(resultados)
df

In [ ]:
xs          = np.linspace(-10, 10, 400)
aprox_vals  = np.array([exp_serie(float(x))[0] for x in xs])
exacto_vals = np.exp(xs)
errores     = np.abs(aprox_vals - exacto_vals) / np.abs(exacto_vals)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(xs, exacto_vals, color='steelblue', lw=2,   label="$e^x$ exacto")
axes[0].plot(xs, aprox_vals,  color='tomato',    lw=1.2, linestyle='--', label="Serie de Taylor")
axes[0].set_title("$e^x$ exacto vs serie de Taylor")
axes[0].set_xlabel("x")
axes[0].legend()
axes[0].grid(True)

axes[1].semilogy(xs, errores + 1e-20, color='seagreen', lw=1.2)
axes[1].set_title("Error relativo de la serie")
axes[1].set_xlabel("x")
axes[1].set_ylabel("Error relativo (escala log)")
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Para x negativos grandes la serie sufre cancelacion catastrofica.
# La alternativa estable es calcular e^|x| y tomar el reciproco.

def exp_serie_estable(x, tol=1e-15, max_iter=1000):
    """
    Para x < 0 calcula exp(-x) y devuelve su reciproco,
    evitando cancelacion catastrofica.
    """
    if x < 0:
        val, n = exp_serie_estable(-x, tol, max_iter)
        return 1.0 / val, n
    return exp_serie(x, tol, max_iter)

# Comparacion para x negativos grandes
valores_neg = [-1, -5, -10, -15, -20]
print(f"{'x':>5}  {'Serie directa':>18}  {'Serie estable':>18}  {'exp(x) real':>18}  {'Err directo':>14}  {'Err estable':>14}")
print("-" * 95)

for x in valores_neg:
    directo, _ = exp_serie(float(x))
    estable, _ = exp_serie_estable(float(x))
    exacto      = np.exp(float(x))
    err_d       = abs(directo - exacto) / abs(exacto)
    err_e       = abs(estable - exacto) / abs(exacto)
    print(f"{x:>5}  {directo:>18.6e}  {estable:>18.6e}  {exacto:>18.6e}  {err_d:>14.6e}  {err_e:>14.6e}")

**Observaciones**

**Criterio de paro:** se detiene cuando el termino actual satisface $|t_n| < \epsilon_{rel} \cdot |S_n|$, aprovechando que los terminos decrecen monotonamente para $x$ moderado. Esto garantiza que se usan exactamente los terminos necesarios sin iteraciones innecesarias.

**Recurrencia del termino:** en lugar de calcular $x^n / n!$ desde cero en cada paso, se actualiza mediante $t_n = t_{n-1} \cdot x/n$, lo que evita desbordamiento y es mas eficiente.

**Inestabilidad para $x \ll 0$:** para valores negativos grandes los terminos de la serie alternan en signo y crecen antes de decrecer, produciendo cancelacion catastrofica. La solucion estable es calcular $e^{|x|}$ (argumento positivo, serie de terminos positivos) y tomar el reciproco.

_(b)_ Sumando en el orden natural, ¿qué criterio de paro debería utilizar?

**Criterio de paro para la serie de $e^x$ sumada en orden natural**

Sumando en orden natural, es decir de $n = 0, 1, 2, \ldots$, el criterio de paro optimo es detener la suma cuando el termino actual $t_n = x^n / n!$ es tan pequeno que ya no modifica la suma parcial $S_n$ en aritmetica de punto flotante:

$fl(S_n + t_n) = S_n$

que en terminos practicos equivale a

$|t_n| < \epsilon_{mach} \cdot |S_n|$

---

**Por que este criterio es correcto para esta serie en particular**

Los terminos de la serie de $e^x$ satisfacen la recurrencia

$t_n = t_{n-1} \cdot \frac{x}{n}$

Para $x > 0$ los terminos son todos positivos y eventualmente decrecen monotonamente a cero (una vez que $n > x$). En ese regimen, si $t_n$ ya es invisible para el acumulador, todos los terminos siguientes $t_{n+1}, t_{n+2}, \ldots$ son aun menores y tambien seran invisibles. El criterio de paro es entonces seguro: no se omite ninguna contribucion representable.

Para $x < 0$ los terminos alternan en signo, pero su magnitud $|t_n|$ sigue la misma recurrencia $|t_n| = |t_{n-1}| \cdot |x|/n$ y tambien decrece eventualmente. El mismo criterio aplicado sobre $|t_n|$ sigue siendo valido.

---

**Criterio que NO se debe usar**

Un criterio incorrecto seria detenerse cuando la diferencia entre sumas parciales consecutivas es pequena:

$|S_n - S_{n-1}| = |t_n| < \delta$

con $\delta$ una tolerancia absoluta fija, porque no toma en cuenta la magnitud de $S_n$. Para $x$ grande $S_n$ puede ser enorme y una tolerancia absoluta pequena detendria la suma demasiado pronto.

---

**Resumen**

El criterio correcto es relativo al acumulador actual:

$|t_n| < \epsilon_{mach} \cdot |S_n|$

Este criterio usa el menor numero de terminos necesarios y es consistente con la precision del sistema de punto flotante empleado.

_(c)_ Prueba un programa para

$$x = \pm 1, \pm 5, \pm 10, \pm 15, \pm 20,$$

y compara tu resultado con la construcción de la función $exp(x).$

In [ ]:
valores_x = [-20, -15, -10, -5, -1, 1, 5, 10, 15, 20]

resultados = []
for x in valores_x:
    directa, n_d = exp_serie(float(x))
    estable, n_e = exp_serie_estable(float(x))
    exacto        = np.exp(float(x))

    err_d = abs(directa - exacto) / abs(exacto)
    err_e = abs(estable - exacto) / abs(exacto)

    resultados.append({
        "x"            : x,
        "Serie directa": directa,
        "Serie estable": estable,
        "exp(x) real"  : exacto,
        "Error directo": err_d,
        "Error estable": err_e,
        "Iter directa" : n_d,
        "Iter estable" : n_e,
    })

df = pd.DataFrame(resultados)
df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

xs_plot  = [str(x) for x in valores_x]
err_d    = df["Error directo"].values
err_e    = df["Error estable"].values

x_idx = np.arange(len(valores_x))
w     = 0.35

axes[0].bar(x_idx - w/2, err_d + 1e-20, w, color='tomato',    label="Serie directa")
axes[0].bar(x_idx + w/2, err_e + 1e-20, w, color='steelblue', label="Serie estable")
axes[0].set_yscale('log')
axes[0].set_xticks(x_idx)
axes[0].set_xticklabels(xs_plot)
axes[0].set_xlabel("x")
axes[0].set_ylabel("Error relativo (escala log)")
axes[0].set_title("Error relativo: serie directa vs estable")
axes[0].axhline(np.finfo(float).eps, color='gray', linestyle=':', label="$\epsilon_{mach}$")
axes[0].legend()
axes[0].grid(True, axis='y')

axes[1].bar(x_idx - w/2, df["Iter directa"].values, w, color='tomato',    label="Serie directa")
axes[1].bar(x_idx + w/2, df["Iter estable"].values, w, color='steelblue', label="Serie estable")
axes[1].set_xticks(x_idx)
axes[1].set_xticklabels(xs_plot)
axes[1].set_xlabel("x")
axes[1].set_ylabel("Iteraciones hasta convergencia")
axes[1].set_title("Numero de terminos usados")
axes[1].legend()
axes[1].grid(True, axis='y')

plt.tight_layout()
plt.show()

**Observaciones**

**Para $x > 0$:** ambas formulas coinciden. Los terminos son positivos y decrecen monotonamente una vez que $n > x$, no hay cancelacion y el error relativo se mantiene cerca de $\epsilon_{mach}$.

**Para $x < 0$ moderado ($x = -1, -5$):** la serie directa aun produce resultados aceptables porque la cancelacion entre terminos positivos y negativos no es severa.

**Para $x \ll 0$ ($x = -10, -15, -20$):** la serie directa sufre cancelacion catastrofica creciente. Los terminos intermedios alcanzan magnitudes del orden de $e^{|x|}$ antes de cancelarse, y el error relativo crece rapidamente hasta perder todos los digitos significativos. Para $x = -20$ el error directo puede superar el $100\%$.

**La serie estable** mantiene error relativo cercano a $\epsilon_{mach}$ para todos los valores de $x$, positivos y negativos, al calcular siempre $e^{|x|}$ con argumento positivo y tomar el reciproco cuando $x < 0$.

**Numero de terminos:** crece con $|x|$ porque se necesitan mas terminos de la serie de Taylor para converger cuando el argumento es grande.

(d) ¿Puedes usar la serie en esta forma para obtener resultados precisos para $x < 0?$ (_Hint:_ $e^{-x} = 1/e^{x}$)

**¿Puede usarse la serie directamente para $x < 0$?**

No, no se puede obtener resultados precisos para $x < 0$ usando la serie directamente en esa forma.

---

**Razon**

Para $x < 0$ los terminos de la serie alternan en signo y sus magnitudes crecen antes de decrecer. El termino de mayor magnitud es aproximadamente $|x|^n / n!$ evaluado en su maximo, que para $x = -20$ alcanza valores del orden de $10^7$. Sin embargo el resultado final $e^{-20} \approx 2 \times 10^{-9}$ es extremadamente pequeno.

Se estan sumando y restando numeros grandes para obtener un resultado pequeno. Cada termino carga un error de redondeo absoluto del orden de $\epsilon_{mach} \cdot |t_n|$, y al cancelarse los terminos ese error permanece mientras el resultado se vuelve minusculo. El error relativo final puede estimarse como

$\text{error relativo} \approx \frac{\epsilon_{mach} \cdot \max_n |t_n|}{|e^x|}  \approx \epsilon_{mach} \cdot e^{|x|} \cdot e^{|x|} = \epsilon_{mach} \cdot e^{2|x|}$

que crece exponencialmente con $|x|$. Para $x = -20$ esto es del orden de $\epsilon_{mach} \cdot 10^{17} \approx 1$, es decir se pierden todos los digitos significativos.

---

**Solucion: usar $e^{-x} = 1/e^x$**

La forma correcta es aprovechar la identidad

$e^x = \frac{1}{e^{-x}}, \quad x < 0$

Se evalua la serie con argumento $-x > 0$, donde todos los terminos son positivos, no hay cancelacion y el error relativo se mantiene cerca de $\epsilon_{mach}$. Finalmente se toma el reciproco. Este es exactamente el enfoque implementado en `exp_serie_estable`.

_(e)_ Puedes reorganizar la serie o reagrupar los términos de alguna manera para obtener resultado mas precisos para $x < 0$?

**Reorganizacion de la serie para $x < 0$**

---

**El problema**

Para $x < 0$ la serie de Taylor de $e^x$ tiene terminos alternantes cuya magnitud crece antes de decrecer. La suma de terminos grandes de signos opuestos produce cancelacion catastrofica, destruyendo los digitos significativos del resultado.

---

**Estrategia 1: argumento positivo y reciproco**

La reorganizacion mas simple y efectiva es aprovechar la identidad

$e^x = \frac{1}{e^{-x}}$

donde $-x > 0$. Se evalua la serie con argumento positivo $-x$, donde todos los terminos son positivos y no hay cancelacion, y se toma el reciproco al final. Esta es la estrategia implementada en `exp_serie_estable` y produce error relativo del orden de $\epsilon_{mach}$ para cualquier $x$.

---

**Estrategia 2: separacion de parte entera**

Si $x = -m - r$ con $m = \lfloor |x| \rfloor$ entero y $0 \leq r < 1$, se puede escribir

$e^x = e^{-m} \cdot e^{-r}$

El factor $e^{-r}$ se evalua con la serie para $r \in [0, 1)$, donde los terminos son pequenos desde el inicio y la convergencia es rapida y precisa. El factor $e^{-m}$ se obtiene como $1/e^m$ calculando $e^m$ con la serie para argumento positivo. El producto final evita operar con argumentos grandes en la serie.

---

**Estrategia 3: reduccion de argumento por cuadrados repetidos**

Se escribe $x = 2^k \cdot s$ donde $s = x / 2^k$ es pequeno (por ejemplo $|s| \leq 1$), se evalua $e^s$ con la serie (convergencia rapida y precisa para $|s|$ pequeno) y se eleva al cuadrado $k$ veces:

$e^x = \left(e^s\right)^{2^k}$

Esto reduce el numero de terminos necesarios y evita la cancelacion asociada a argumentos grandes.

---

**Conclusion**

Las tres estrategias son superiores a sumar la serie directamente para $x < 0$. En la practica la **Estrategia 1** es la mas simple e igualmente precisa: nunca se debe evaluar la serie de Taylor de $e^x$ directamente para $x \ll 0$.

**1.10** Escribe un programa que resuelva la ecuación cuadrática $ax^2 + bx + c = 0$ using the standard quadratic formula

$$x = \frac{-b \pm \sqrt{b^2 - 4ac}}{2a}$$

o la formula alternativa

$$x = \frac{2c}{-b \mp \sqrt{b^2 - 4ac}}$$

Su programa debe aceptar valores de los coeficientes $a$, $b$
y $c$ como entrada y generar las dos raíces de la ecuación como salida. Su programa debe detectar cuándo las raíces son imaginarias, pero no necesita usar aritmética compleja explícitamente. Debe evitar desbordamientos subdesbordamientos y cancelaciones innecesarios. ¿Cuándo
debería usar cada una de las dos fórmulas? Intente que su
programa sea robusto cuando se le den valores de entrada
inusuales. Cualquier raíz que esté dentro del rango del sistema de punto flotante debe calcularse con precisión,  incluso si la otra está fuera de rango. Pruebe su programa
utilizando los siguientes valores para los coeficientes:

| $a$ | $b$ | $c$ |
|-----|-----|-----|
| $6$ | $5$ | $-4$ |
| $6 \times 10^{30}$ | $5 \times 10^{30}$ | $-4 \times 10^{30}$ |
| $0$ | $1$ | $1$ |
| $1$ | $-10^{5}$ | $1$ |
| $1$ | $-4$ | $3.999999$ |
| $10^{-30}$ | $-10^{30}$ | $10^{30}$ |

In [ ]:
def cuadratica_robusta(a, b, c):
    """
    Resuelve ax^2 + bx + c = 0 de forma numericamente robusta.
    Usa la formula estandar o la alternativa segun cual evite cancelacion.
    Detecta raices imaginarias, ecuacion lineal (a=0) y casos degenerados.
    """

    # --- Caso degenerado: a = 0 ---
    if a == 0.0:
        if b == 0.0:
            if c == 0.0:
                return "Infinitas soluciones (0 = 0)", None
            else:
                return "Sin solucion (ecuacion inconsistente)", None
        else:
            x = -c / b
            return x, None

    # --- Discriminante: calcular con cuidado ---
    # Escalar para evitar overflow en b^2 o 4ac
    # Se normaliza dividiendo por a
    # b^2 - 4ac = a^2 * ((b/a)^2 - 4*(c/a))
    # pero es mas seguro trabajar directamente con float64

    discriminante = b*b - 4.0*a*c

    # --- Raices imaginarias ---
    if discriminante < 0.0:
        re   = -b / (2.0 * a)
        im   =  np.sqrt(-discriminante) / (2.0 * abs(a))
        signo = "+" if a > 0 else "-"
        return complex(re,  im), complex(re, -im)

    sqrt_d = np.sqrt(discriminante)

    # --- Eleccion de formula para evitar cancelacion ---
    # Si b > 0: -b es negativo, usar formula estandar para x1 (suma con -sqrt_d)
    #           y formula alternativa para x2
    # Si b < 0: -b es positivo, usar formula estandar para x1 (suma con +sqrt_d)
    #           y formula alternativa para x2
    # En ambos casos: la raiz sin cancelacion es la que suma terminos del mismo signo

    if b >= 0.0:
        # -b - sqrt_d es la suma de dos negativos: sin cancelacion
        q  = -0.5 * (b + sqrt_d)
    else:
        # -b + sqrt_d es la suma de dos positivos: sin cancelacion
        q  = -0.5 * (b - sqrt_d)

    # q = (-b - sign(b)*sqrt_d) / 2
    # x1 = q / a  (formula estandar para la raiz sin cancelacion)
    # x2 = c / q  (formula alternativa para la otra raiz)

    if q == 0.0:
        # b = 0 y discriminante = 0: raiz doble en 0, c debe ser 0
        x1, x2 = 0.0, 0.0
    else:
        x1 = q / a
        x2 = c / q

    return x1, x2


# --- Casos de prueba ---
casos = [
    (6,       5,       -4      ),
    (6e30,    5e30,    -4e30   ),
    (0,       1,        1      ),
    (1,      -1e5,      1      ),
    (1,      -4,        3.999999),
    (1e-30,  -1e30,     1e30   ),
]

print(f"{'a':>12} {'b':>12} {'c':>12}  {'x1':>25} {'x2':>25}")
print("-" * 92)
for a, b, c in casos:
    x1, x2 = cuadratica_robusta(a, b, c)
    print(f"{a:>12.3g} {b:>12.3g} {c:>12.3g}  {str(x1):>25} {str(x2):>25}")

In [ ]:
# --- Verificacion: sustituir raices en la ecuacion original ---
print("\nVerificacion |ax^2 + bx + c| para cada raiz real:")
print(f"{'a':>10} {'b':>10} {'c':>10}  {'|f(x1)|':>15} {'|f(x2)|':>15}")
print("-" * 65)

for a, b, c in casos:
    x1, x2 = cuadratica_robusta(a, b, c)
    for xi, etiq in [(x1, "x1"), (x2, "x2")]:
        if xi is None or isinstance(xi, str):
            continue
        if isinstance(xi, complex):
            residuo = abs(a*xi**2 + b*xi + c)
        else:
            residuo = abs(a*xi**2 + b*xi + c)
        print(f"{a:>10.3g} {b:>10.3g} {c:>10.3g}  {etiq}: |f(x)| = {residuo:.4e}")

**Criterio de eleccion de formula**

La clave es evitar restar dos numeros casi iguales en el numerador. Se define

$q = -\frac{1}{2}\left(b + \text{sgn}(b)\sqrt{b^2 - 4ac}\right)$

y se obtienen las dos raices como

$x_1 = \frac{q}{a}, \qquad x_2 = \frac{c}{q}$

La raiz $x_1$ usa la formula estandar aplicada al termino sin cancelacion, y $x_2$ usa la formula alternativa basada en la relacion de Vieta $x_1 x_2 = c/a$. Asi se garantiza que ninguna raiz se obtiene por resta de cantidades casi iguales.

**Casos notables**

- $a = 0$: la ecuacion es lineal y se resuelve directamente.
- Discriminante negativo: se reportan las raices complejas conjugadas.
- Coeficientes muy grandes ($10^{30}$): la normalizacion implicita en la eleccion de $q$ evita overflow.
- $b^2 \gg 4ac$ ($a=1, b=-10^5, c=1$): sin la formula alternativa, la raiz pequena perderia todos sus digitos significativos.
- Discriminante casi cero ($a=1, b=-4, c=3.999999$): raices casi iguales, sensibles a perturbaciones en los coeficientes.